# 04 — Threading & Multiprocessing

`🔴 Advanced`

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaIslamFahad/python-for-ai-engineers/blob/main/03_Advanced/04_multithreading_multiprocessing.ipynb)

## 📌 What is it?
**Threading**: multiple threads share memory — good for I/O-bound work (limited by GIL for CPU work). **Multiprocessing**: separate processes with separate memory — bypasses GIL, good for CPU-bound work.

## 🧠 Why AI Engineers need this
Data loading pipelines, parallel model evaluation, batch API calls, and preprocessing on multi-core machines all use concurrency.

In [ ]:
# ── concurrent.futures (the modern, recommended API) ──
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import time

def call_llm(prompt: str) -> str:
    """Simulate an API call (I/O bound)."""
    time.sleep(0.1)
    return f"Response: {prompt[:20]}..."

prompts = [f"Question {i}: Explain topic {i}" for i in range(10)]

# Sequential
start = time.time()
results = [call_llm(p) for p in prompts]
print(f"Sequential: {time.time()-start:.2f}s")

# Threaded (great for I/O like API calls)
start = time.time()
with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(call_llm, prompts))
print(f"Threaded (5 workers): {time.time()-start:.2f}s")
print(f"Results: {len(results)}")

In [ ]:
# ── MULTIPROCESSING for CPU-bound work ──
from concurrent.futures import ProcessPoolExecutor
import time

def cpu_heavy(n: int) -> int:
    """CPU-bound: count primes up to n."""
    count = 0
    for i in range(2, n):
        if all(i % j != 0 for j in range(2, int(i**0.5)+1)):
            count += 1
    return count

numbers = [10000, 15000, 12000, 8000]

# Sequential
start = time.time()
results = [cpu_heavy(n) for n in numbers]
print(f"Sequential: {time.time()-start:.2f}s → {results}")

# Parallel processes (bypasses GIL for true parallelism)
start = time.time()
with ProcessPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(cpu_heavy, numbers))
print(f"Multiprocess: {time.time()-start:.2f}s → {results}")

In [ ]:
# ── THREADING with shared state ──
import threading

class SafeCounter:
    """Thread-safe counter using a Lock."""
    def __init__(self):
        self._count = 0
        self._lock = threading.Lock()
    
    def increment(self):
        with self._lock:   # only one thread at a time
            self._count += 1
    
    @property
    def count(self):
        return self._count

counter = SafeCounter()

def worker(n: int):
    for _ in range(n):
        counter.increment()

threads = [threading.Thread(target=worker, args=(100,)) for _ in range(10)]
for t in threads: t.start()
for t in threads: t.join()

print(f"Expected: 1000, Got: {counter.count}")

## ⚠️ Common Mistakes
```python
# ❌ Using threading for CPU-heavy Python code
# The GIL means only one thread runs Python at a time
# For CPU work → use ProcessPoolExecutor or asyncio

# ❌ Unsynchronized shared state
count = 0
def bad_worker():
    global count
    count += 1   # NOT thread-safe — race condition!

# ✅ Use a Lock or thread-safe data structures
# ✅ Or better: use concurrent.futures and avoid shared state
```

## 🔗 What's Next?
[05 — Design Patterns →](05_design_patterns.ipynb)